Titanic kNN - k and distance metric optimization

In [4]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [5]:
# Download latest version
path = kagglehub.dataset_download("yasserh/titanic-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'titanic-dataset' dataset.
Path to dataset files: /kaggle/input/titanic-dataset


In [6]:
df = pd.read_csv(path+"/Titanic-Dataset.csv")

df.count()

,0
PassengerId,891
Survived,891
Pclass,891
Name,891
Sex,891
Age,714
SibSp,891
Parch,891
Ticket,891
Fare,891


In [7]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']
X = df[features]
y = df['Survived']

numeric_features = ['Pclass', 'Age', 'Fare', 'SibSp', 'Parch']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_features = ['Sex']
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(drop='if_binary'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier(n_neighbors=5))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf.fit(X_train, y_train)

accuracy = clf.score(X_test, y_test)
print(f"Titanic Accuracy (One-Hot Encoded): {accuracy:.2%}")

Titanic Accuracy (One-Hot Encoded): 81.01%


## Grid Search Cross-Validation for `k` and Distance Metric

In [14]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

param_grid = {
    'classifier__n_neighbors': list(range(5, 31)),
    'classifier__metric': ['euclidean', 'manhattan', 'minkowski',
                           'chebyshev', 'hamming']
}

knn_classifier = KNeighborsClassifier()

pipe_for_grid = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', knn_classifier)
])

grid_search = GridSearchCV(pipe_for_grid,
                           param_grid,
                           cv=5,
                           scoring='accuracy',
                           n_jobs=-1,
                           verbose=1)

In [22]:
import time

print("Grid Search Cross-Validation:")
start_time = time.time()
grid_search.fit(X_train, y_train)
end_time = time.time()
elapsed_time = end_time - start_time

print(f"\nBest parameters found: {grid_search.best_params_}")
print(f"Best cross-validation accuracy: {grid_search.best_score_:.2%}")
print(f"Grid Search execution time: {elapsed_time:.2f} seconds")

best_model = grid_search.best_estimator_
test_accuracy = best_model.score(X_test, y_test)
print(f"Test set accuracy with best parameters: {test_accuracy:.2%}")

Grid Search Cross-Validation:
Fitting 5 folds for each of 130 candidates, totalling 650 fits

Best parameters found: {'classifier__metric': 'manhattan', 'classifier__n_neighbors': 22}
Best cross-validation accuracy: 82.02%
Grid Search execution time: 15.98 seconds
Test set accuracy with best parameters: 82.12%


## Bayesian Optimization for `k` and Distance Metric

In [18]:
pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 4.3 MB/s eta 0:00:00


In [19]:
from skopt import gp_minimize
from skopt.space import Integer, Categorical
from skopt.utils import use_named_args
from sklearn.model_selection import cross_val_score

dimensions_bo = [
    Integer(5, 30, name='n_neighbors'),
    Categorical(['euclidean', 'manhattan', 'minkowski', 'chebyshev', 'hamming'], name='metric')
]

@use_named_args(dimensions_bo)
def objective_bo(n_neighbors, metric):
    clf_optimized_bo = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', KNeighborsClassifier(n_neighbors=n_neighbors, metric=metric))
    ])

    accuracy = cross_val_score(clf_optimized_bo, X_train, y_train, cv=5, n_jobs=-1, scoring='accuracy').mean()
    return -accuracy

In [23]:
print("Bayesian optimization for n_neighbors and metric:")
start_time_bo = time.time()
search_result_bo = gp_minimize(func=objective_bo,
                               dimensions=dimensions_bo,
                               n_calls=30,
                               random_state=42,
                               n_initial_points=10,
                               verbose=True)
end_time_bo = time.time()
elapsed_time_bo = end_time_bo - start_time_bo

best_n_neighbors_bo = search_result_bo.x[0]
best_metric_bo = search_result_bo.x[1]
best_neg_accuracy_bo = search_result_bo.fun
best_accuracy_bo = -best_neg_accuracy_bo

print(f"\nBest n_neighbors found by BO: {best_n_neighbors_bo}")
print(f"Best metric found by BO: {best_metric_bo}")
print(f"Best cross-validation accuracy by BO: {best_accuracy_bo:.2%}")
print(f"Bayesian Optimization execution time: {elapsed_time_bo:.2f} seconds")

best_model_bo = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier(n_neighbors=best_n_neighbors_bo, metric=best_metric_bo))
])
best_model_bo.fit(X_train, y_train)
test_accuracy_bo = best_model_bo.score(X_test, y_test)
print(f"Test set accuracy with best BO parameters: {test_accuracy_bo:.2%}")

Bayesian optimization for n_neighbors and metric:
Iteration No: 1 started. Evaluating function at random point.
Iteration No: 1 ended. Evaluation done at random point.
Time taken: 0.2921
Function value obtained: -0.8174
Current minimum: -0.8174
Iteration No: 2 started. Evaluating function at random point.
Iteration No: 2 ended. Evaluation done at random point.
Time taken: 0.2957
Function value obtained: -0.8174
Current minimum: -0.8174
Iteration No: 3 started. Evaluating function at random point.
Iteration No: 3 ended. Evaluation done at random point.
Time taken: 0.2773
Function value obtained: -0.7977
Current minimum: -0.8174
Iteration No: 4 started. Evaluating function at random point.
Iteration No: 4 ended. Evaluation done at random point.
Time taken: 0.2583
Function value obtained: -0.8146
Current minimum: -0.8174
Iteration No: 5 started. Evaluating function at random point.
Iteration No: 5 ended. Evaluation done at random point.
Time taken: 0.2447
Function value obtained: -0.7725


/usr/local/lib/python3.12/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(21), np.str_('manhattan')] before, using random point [np.int64(15), 'euclidean']
  warnings.warn(


Iteration No: 15 ended. Search finished for the next optimal point.
Time taken: 1.5008
Function value obtained: -0.8005
Current minimum: -0.8174
Iteration No: 16 started. Searching for the next optimal point.
Iteration No: 16 ended. Search finished for the next optimal point.
Time taken: 0.5801
Function value obtained: -0.7978
Current minimum: -0.8174
Iteration No: 17 started. Searching for the next optimal point.
Iteration No: 17 ended. Search finished for the next optimal point.
Time taken: 0.5148
Function value obtained: -0.7949
Current minimum: -0.8174
Iteration No: 18 started. Searching for the next optimal point.
Iteration No: 18 ended. Search finished for the next optimal point.
Time taken: 0.6127
Function value obtained: -0.8118
Current minimum: -0.8174
Iteration No: 19 started. Searching for the next optimal point.
Iteration No: 19 ended. Search finished for the next optimal point.
Time taken: 0.5801
Function value obtained: -0.8202
Current minimum: -0.8202
Iteration No: 20 st

/usr/local/lib/python3.12/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(22), np.str_('manhattan')] before, using random point [np.int64(6), 'euclidean']
  warnings.warn(


Iteration No: 20 ended. Search finished for the next optimal point.
Time taken: 0.5709
Function value obtained: -0.7865
Current minimum: -0.8202
Iteration No: 21 started. Searching for the next optimal point.


/usr/local/lib/python3.12/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(22), np.str_('manhattan')] before, using random point [np.int64(24), 'hamming']
  warnings.warn(


... (3 outputs left out)

/usr/local/lib/python3.12/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(21), np.str_('manhattan')] before, using random point [np.int64(3), 'minkowski']
  warnings.warn(


Iteration No: 29 ended. Search finished for the next optimal point.
Time taken: 0.5975
Function value obtained: -0.8132
Current minimum: -0.8202
Iteration No: 30 started. Searching for the next optimal point.


/usr/local/lib/python3.12/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(22), np.str_('manhattan')] before, using random point [np.int64(14), 'manhattan']
  warnings.warn(


Iteration No: 30 ended. Search finished for the next optimal point.
Time taken: 0.6738
Function value obtained: -0.8146
Current minimum: -0.8202

Best n_neighbors found by BO: 22
Best metric found by BO: manhattan
Best cross-validation accuracy by BO: 82.02%
Bayesian Optimization execution time: 19.37 seconds
Test set accuracy with best BO parameters: 82.12%


Both GridSearchCV and Bayesian Optimization techniques achieved the same local optimal hyperparameters (k=22m distance metric = 'manhattan') with final test accuracy 82.12%. GridSearch was sligthly faster, possibly due to small search space.